<a href="https://colab.research.google.com/github/MohammedZaid-AI/Smart-Coffee-Recommender/blob/master/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

In [30]:
data=pd.read_csv("/content/Coffe_sales_updated.csv")

In [31]:
data.head()

,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,01-03-2024,15:50.5
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,01-03-2024,19:22.5
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,01-03-2024,20:18.1
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,01-03-2024,46:33.0
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,01-03-2024,48:14.6


In [34]:
data.shape

(3547, 11)

In [35]:
data.isnull().sum()

,0
hour_of_day,0
cash_type,0
money,4
coffee_name,0
Time_of_Day,0
Weekday,0
Month_name,0
Weekdaysort,0
Monthsort,0
Date,0


In [66]:
data['money']=data['money'].fillna(data['money'].mean())

Here’s your content reorganized into a clean, structured format:

---

## **Handling Missing Data**

### **1. Directional Filling (Time Series / Sequential Data)**

Used when data has an inherent order (e.g., stock prices, sensor readings).

* **Forward Fill (ffill)**
  Propagates the last valid observation forward to fill missing values.

  ```python
  df.fillna(method='ffill')
  ```

* **Backward Fill (bfill)**
  Uses the next valid observation to fill missing values backward.

  ```python
  df.fillna(method='bfill')
  ```

---

### **2. Statistical Imputation**

Replaces missing values with summary statistics to preserve dataset size and avoid dropping data.

* **Mean (Average)**
  Suitable for numerical data without extreme outliers.

  ```python
  df['column'].fillna(df['column'].mean())
  ```

* **Median**
  More robust for skewed numerical data or when outliers are present.

  ```python
  df['column'].fillna(df['column'].median())
  ```

* **Mode (Most Frequent Value)**
  Best for categorical data (e.g., strings).

  ```python
  df['column'].fillna(df['column'].mode()[0])
  ```

---

If you want, I can also turn this into a comparison table or add when to use each method in real scenarios.


In [67]:
data.isnull().sum()

,0
hour_of_day,0
cash_type,0
money,0
coffee_name,0
Time_of_Day,0
Weekday,0
Month_name,0
Weekdaysort,0
Monthsort,0
Date,0


In [71]:
data['day_of_the_week']=pd.to_datetime(data['Date'], dayfirst=True).dt.dayofweek

In [69]:
def get_time_of_day(hour):
  if 5 <= hour <= 12:
    return "Morning"
  elif 13 <= hour <= 16:
    return "Afternoon"
  else:
    return "Evening"

In [72]:
data['hour']=data['hour_of_day']
data['time_of_day']=data['hour'].apply(get_time_of_day)

In [73]:
data.head()

,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time,day_of_the_week,hour,time_of_day
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,01-03-2024,15:50.5,4,10,Morning
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,01-03-2024,19:22.5,4,12,Morning
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,01-03-2024,20:18.1,4,12,Morning
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,01-03-2024,46:33.0,4,13,Afternoon
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,01-03-2024,48:14.6,4,13,Afternoon


In [74]:
data.shape

(3547, 14)

In [75]:
x=data[['day_of_the_week','time_of_day']]
y=data['coffee_name']

In [76]:
le_time = LabelEncoder()
le_day = LabelEncoder()
le_coffee = LabelEncoder()
x.loc[:,'day_of_the_week']=le_day.fit_transform(x['day_of_the_week'])
x.loc[:,'time_of_day']=le_time.fit_transform(x['time_of_day'])
y=le_coffee.fit_transform(y)

/tmp/ipykernel_5026/1592106926.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[4 4 4 ... 6 6 6]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  x.loc[:,'day_of_the_week']=le_day.fit_transform(x['day_of_the_week'])


In [77]:
display(x.head())

,day_of_the_week,time_of_day
0,4,2
1,4,2
2,4,2
3,4,0
4,4,0


In [78]:
model=RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x,y)


RandomForestClassifier(random_state=42)

In [79]:
def predict_coffee(day_of_the_week, time_of_day):
  day_of_the_week_encoded=le_day.transform([day_of_the_week])[0]
  time_of_day_encoded=le_time.transform([time_of_day])[0]

  predict=model.predict([[day_of_the_week_encoded, time_of_day_encoded]])
  return le_coffee.inverse_transform(predict)[0]

In [80]:
joblib.dump(model, "coffee_model.pkl")

['coffee_model.pkl']

In [81]:
joblib.dump(le_day, "le_day.pkl")

['le_day.pkl']

In [55]:
joblib.dump(le_time, "le_time.pkl")

['le_time.pkl']

In [56]:
joblib.dump(le_coffee, "le_coffee.pkl")

['le_coffee.pkl']